# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their associated field (column) ids
print("Available Record Sets and Their Fields/Columns:")

for record_set in dataset.record_sets:
    print(f"- RecordSet: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    print("  Columns/Fields:")
    field_list = record_set.get('field', [])
    if isinstance(field_list, dict):
        field_list = [field_list]
    for field in field_list:
        field_id = field['@id'] if isinstance(field, dict) else field
        field_obj = dataset.field(field_id)
        print(f"    - {field_id} ({field_obj.data_type if hasattr(field_obj, 'data_type') else ''})")
    print()
if not dataset.record_sets:
    print("No RecordSets declared in the @recordSet field. Searching for inferred record sets from resources...")
    # Try to show what files are present
    for file_obj in dataset.file_objects:
        print(f"- FileObject: {file_obj['@id']}")
        print(f"  ContentUrl: {file_obj.get('contentUrl','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Try to extract a records set by id.
# As the official Croissant @recordSet declaration is empty, we'll try to find a file which has tabular data.

# Find first FileObject with a CSV/TSV
main_file_id = None
for file_obj in dataset.file_objects:
    cformat = file_obj.get('encodingFormat','')
    if cformat and ('csv' in cformat or 'tsv' in cformat):
        main_file_id = file_obj['@id']
        break
if not main_file_id:
    # Fallback to first file object
    if len(dataset.file_objects) > 0:
        main_file_id = dataset.file_objects[0]['@id']

# List the records for this main file
print(f"Extracting records from file object with @id: {main_file_id}")
try:
    records = list(dataset.records(file_object=main_file_id))
    df = pd.DataFrame(records)
    print("Available columns:", df.columns.tolist())
    display(df.head())
except Exception as e:
    print(f"Could not extract records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll identify a numeric field (like protein abundance, "MW", etc) and run typical workflows.
# We'll attempt to select a common numeric column.
numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # Try to coerce numeric for common field names
    for name in ['MW', 'Abundance', 'abundance', 'score', 'Coverage', 'peptide', 'count', 'Score']:
        if name in df.columns:
            try:
                df[name] = pd.to_numeric(df[name], errors='coerce')
                numeric_candidates.append(name)
            except Exception:
                continue
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}")
    threshold = df[numeric_field].quantile(0.75) if not pd.isnull(df[numeric_field].max()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    # Normalization
    norm_col = f'{numeric_field}_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())
    # Try grouping by a string/categorical field
    cat_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if cat_candidates:
        group_field = cat_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if we found a usable numeric field
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If we found a cat for group_field, show boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(12,4))
        order = df[group_field].value_counts().iloc[:10].index.tolist()  # most common categories
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(order)], order=order)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no numeric field available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!--
During our exploration of the Mass Spectrometry Analysis dataset, we loaded and surveyed available fields from the main record set (file resource). We identified numeric fields such as protein molecular weight and performed basic filtering and normalization, grouping (if possible), and visualized their distribution. This workflow demonstrates standard EDA steps for datasets structured via the Croissant schema and loaded with `mlcroissant`.
-->